# FP8 W8A8 양자화 (대안 경로)

## 목적
L4 GPU의 FP8 텐서코어를 활용한 W8A8 양자화로 다른 PerfNorm/SpeedNorm 트레이드오프 탐색.

## 기대값
- **PerfNorm**: ~0.97-0.99 (8비트는 4비트보다 정밀도 보존 우수)
- **SpeedNorm**: 불확실 (FP8 텐서코어 vs Marlin W4A16)
- **모델 크기**: ~1.7GB (W4A16의 1.4GB보다 큼)

## 리스크
- memory-bandwidth-bound 시나리오에서 W4A16 Marlin이 더 빠를 수 있음
- FP8이 L4에서 제대로 동작하는지 확인 필요

---

# 1. Import

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name}")
    print(f"Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
    if compute_cap[0] >= 8 and compute_cap[1] >= 9:
        print("\u2705 FP8 지원 (Ada Lovelace)")
    elif compute_cap[0] >= 9:
        print("\u2705 FP8 지원 (Hopper)")
    else:
        print("\u26a0\ufe0f FP8 미지원 GPU - W8A8 INT8로 대체 가능")
else:
    print("\u26a0\ufe0f CPU 모드 - FP8 양자화는 GPU 필요")
print("\n\u2705 Import 완료!")

# 2. 설정

In [ ]:
# ============================================================================
# 모델 설정
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = "./model_fp8"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# FP8 양자화 설정
# ============================================================================
# 방법 A: 단순 FP8 (QuantizationModifier)
# 방법 B: FP8 + Dynamic Activation Quantization
METHOD = "A"  # A 또는 B 선택

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print(f"FP8 양자화 (방법 {METHOD})")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_LEN: {MAX_SEQUENCE_LENGTH}")
print("=" * 60)

# 3. 모델 로드

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델/토크나이저 로드 완료")

# 4. 데이터셋 로드

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

# 5. FP8 양자화

In [ ]:
print(f"[INFO] FP8 양자화 시작 (방법 {METHOD})")

if METHOD == "A":
    # 방법 A: 단순 FP8 Weight-Only
    recipe = [
        QuantizationModifier(
            targets="Linear",
            scheme="FP8",
            ignore=["lm_head"],
        )
    ]
    print("  방법 A: FP8 Weight-Only (Static)")
    print("  - weights: FP8 E4M3")
    print("  - activations: FP16 (변경 없음)")

elif METHOD == "B":
    # 방법 B: FP8 W8A8 (Weight + Dynamic Activation)
    recipe = [
        QuantizationModifier(
            targets="Linear",
            scheme="FP8_DYNAMIC",
            ignore=["lm_head"],
        )
    ]
    print("  방법 B: FP8 W8A8 Dynamic")
    print("  - weights: FP8 E4M3 (Static)")
    print("  - activations: FP8 E4M3 (Dynamic, per-token)")

if torch.cuda.is_available():
    print("\n\U0001f680 GPU 모드\n")
else:
    print("\n\u23f3 CPU 모드\n")

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print(f"\n[INFO] FP8 양자화 완료!")

# 6. 모델 저장

In [ ]:
print(f"[INFO] 모델 저장: {OUT_DIR}")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("FP8 모델 크기 비교")
print("=" * 60)
print(f"  원본 모델 (FP16):   {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  FP8 모델:           {quantized_size_gb:.2f} GB")
print(f"  W4A16 모델 (V13):   ~1.42 GB")
print(f"  압축률:             {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

if quantized_size_gb > 1.5:
    print("\n\u26a0\ufe0f FP8 모델이 W4A16보다 큼 → SpeedNorm 손실 가능")
    print("  Score = 0.5*PerfNorm + 0.5*SpeedNorm")
    print("  PerfNorm 향상이 SpeedNorm 손실을 상회해야 유리")

# 7. 제출 파일 생성

In [ ]:
submit_dir = "./submit"
os.makedirs(submit_dir, exist_ok=True)

zip_name = f"submit_fp8_{METHOD.lower()}"
zip_path = os.path.join(submit_dir, zip_name)

print(f"[INFO] {zip_name}.zip 생성 중...")

if os.path.exists(f"{zip_path}.zip"):
    os.remove(f"{zip_path}.zip")

shutil.make_archive(
    base_name=zip_path,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_path}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_path}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("\u2705 용량 제한 충족 (\u2264 10GB)")
else:
    print("\u274c 용량 초과!")

# 8. 제출용 submit.zip 생성 (선택)

In [ ]:
MAKE_FINAL = False  # True로 변경하면 실행

if MAKE_FINAL:
    final_model_dir = "./model"
    if os.path.exists(final_model_dir):
        shutil.rmtree(final_model_dir)
    shutil.copytree(OUT_DIR, final_model_dir)
    print(f"[INFO] {OUT_DIR} → {final_model_dir} 복사 완료")

    if os.path.exists("submit.zip"):
        os.remove("submit.zip")
    shutil.make_archive(
        base_name="submit",
        format="zip",
        root_dir=".",
        base_dir="model",
    )
    final_zip_size = os.path.getsize("submit.zip") / 1e9
    print(f"[INFO] submit.zip 생성 완료 ({final_zip_size:.2f} GB)")
else:
    print("[INFO] MAKE_FINAL=False → 건너뜀")

---

# 결과 비교

| 방법 | 모델 크기 | PerfNorm | SpeedNorm | Score |
|------|----------|----------|-----------|-------|
| V13 (W4A16 + L0,L29 FP16) | ~1.42 GB | ? | ? | 최고점 |
| FP8 방법 A (Weight-Only) | | | | |
| FP8 방법 B (W8A8 Dynamic) | | | | |

### 판단 기준
- FP8 Score > V13 Score → FP8 채택
- FP8 Score < V13 Score → V13 유지, 전략 1/2에 집중